In [32]:
# Project: Bellabeat Case Study Capstone
# Objective: Analyze smart device usage data to identify behavioral trends and generate
# marketing insights for the Bellabeat Leaf product.

# Datasets:
# - dailyActivity_merged.csv
# - sleepDay_merged.csv

# Outputs:
# - df_master_33_users.csv (activity dataset with optional sleep data)
# - df_sleep_master_24_users.csv (subset for sleep analysis)

# Author: Diego Fernandez
# Date: 2026


In [33]:
# Pull up databases and convert variables

import pandas as pd

df_daily = pd.read_csv("dailyActivity_merged.csv")
df_sleep = pd.read_csv("sleepDay_merged.csv", dtype={"SleepDay": "string"})

In [34]:
# Change to date format and normalize

df_daily["ActivityDate"] = pd.to_datetime(df_daily["ActivityDate"], errors="coerce")
df_daily["Date"] = df_daily ["ActivityDate"].dt.normalize()

print("Daily users:", df_daily["Id"].nunique())
print("Daily rows:", len(df_daily))
print("Daily ActivityDate NaT:", df_daily["ActivityDate"].isna().sum())

Daily users: 33
Daily rows: 940
Daily ActivityDate NaT: 0


In [35]:
# Test Normalization

print(df_sleep["SleepDay"].head(10))

0    4/12/2016 12:00:00 AM
1    4/13/2016 12:00:00 AM
2    4/15/2016 12:00:00 AM
3    4/16/2016 12:00:00 AM
4    4/17/2016 12:00:00 AM
5    4/19/2016 12:00:00 AM
6    4/20/2016 12:00:00 AM
7    4/21/2016 12:00:00 AM
8    4/23/2016 12:00:00 AM
9    4/24/2016 12:00:00 AM
Name: SleepDay, dtype: string


In [36]:
# Parsing and test

s = df_sleep["SleepDay"].str.strip()

parsed_time = pd.to_datetime(s, format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
parsed_date = pd.to_datetime(s, format="%m/%d/%y", errors="coerce")

df_sleep["SleepDay_dt"] = parsed_time.fillna(parsed_date)
df_sleep["Date"] = df_sleep["SleepDay_dt"].dt.normalize()

print("Sleep users:", df_sleep["Id"].nunique())
print("Sleep rows:", len(df_sleep))
print("SleepDay_dt NaT:", df_sleep["SleepDay_dt"].isna().sum())
                             

Sleep users: 24
Sleep rows: 413
SleepDay_dt NaT: 0


In [37]:
# Merge daily and test

df_master_33 = df_daily.merge(
    df_sleep[["Id","Date","TotalMinutesAsleep","TotalTimeInBed","TotalSleepRecords"]],
    on=["Id","Date"],
    how="left"
)

print("Mater 33 users:", df_master_33["Id"].nunique())
print("master 33 rows:", len(df_master_33))
print("Rows with sleep:", df_master_33["TotalMinutesAsleep"].notna().sum())

Mater 33 users: 33
master 33 rows: 943
Rows with sleep: 413


In [38]:
# Merge sleep and test

df_sleep_master_24 = df_daily.merge(
    df_sleep[["Id","Date","TotalMinutesAsleep","TotalTimeInBed","TotalSleepRecords"]],
    on=["Id","Date"],
    how="inner"
)

print("Sleep subset users:", df_sleep_master_24["Id"].nunique())
print("Sleep subset rows:", len(df_sleep_master_24))   
print("Rows with sleep:", df_sleep_master_24["TotalMinutesAsleep"].notna().sum())

Sleep subset users: 24
Sleep subset rows: 413
Rows with sleep: 413


In [39]:
# Save new databases

df_master_33.to_csv("df_master_33_users.csv", index=False)
df_sleep_master_24.to_csv("df_sleep_master_24_users.csv", index=False)

In [40]:
# Sample database

df_master_33.head()

,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,Date,TotalMinutesAsleep,TotalTimeInBed,TotalSleepRecords
0,1503960366,2016-04-12,13162,8.50,8.50,0.0,1.88,0.55,6.06,0.0,25,13,328,728,1985,2016-04-12,327.0,346.0,1.0
1,1503960366,2016-04-13,10735,6.97,6.97,0.0,1.57,0.69,4.71,0.0,21,19,217,776,1797,2016-04-13,384.0,407.0,2.0
2,1503960366,2016-04-14,10460,6.74,6.74,0.0,2.44,0.40,3.91,0.0,30,11,181,1218,1776,2016-04-14,NaN,NaN,NaN
3,1503960366,2016-04-15,9762,6.28,6.28,0.0,2.14,1.26,2.83,0.0,29,34,209,726,1745,2016-04-15,412.0,442.0,1.0
4,1503960366,2016-04-16,12669,8.16,8.16,0.0,2.71,0.41,5.04,0.0,36,10,221,773,1863,2016-04-16,340.0,367.0,2.0


In [41]:
# Identify number of duplicates

print("Daily Duplicates:", df_daily.duplicated(["Id","Date"]).sum())
print("Sleep duplicates:", df_sleep.duplicated(["Id","Date"]).sum())

Daily Duplicates: 0
Sleep duplicates: 3


In [42]:
# Drop duplicates

df_sleep = df_sleep.drop_duplicates(["Id","Date"])

In [43]:
# Re-merge daily

df_master_33 = df_daily.merge(
    df_sleep[["Id","Date","TotalMinutesAsleep","TotalTimeInBed","TotalSleepRecords"]],
    on=["Id","Date"],
    how="left"
)

In [44]:
# Check to make sure duplicates are gone

print("Daily Duplicates:", df_daily.duplicated(["Id","Date"]).sum())
print("Sleep duplicates:", df_sleep.duplicated(["Id","Date"]).sum())

Daily Duplicates: 0
Sleep duplicates: 0


In [45]:
# Round distance to the second decimal

cols = [
    "TotalDistance",
    "TrackerDistance",
    "VeryActiveDistance",
    "ModeratelyActiveDistance",
    "LightActiveDistance"
]

df_master_33[cols] = df_master_33[cols].round(2)

In [46]:
# Re-save merge databases

df_master_33.to_csv("df_master_33_users.csv", index=False)
df_sleep_master_24.to_csv("df_sleep_master_24_users.csv", index=False)

In [47]:
# test rows and confirm clean data

print("Master 33 users:", df_master_33["Id"].nunique())
print("Master 33 rows:", len(df_master_33))
print("Rows with sleep:", df_master_33["TotalMinutesAsleep"].notna().sum())

Master 33 users: 33
Master 33 rows: 940
Rows with sleep: 410


In [48]:
# Create new column (day_of_wee)

df_master_33["day_of_week"] = df_master_33["Date"].dt.day_name()
df_master_33.groupby("day_of_week")["TotalSteps"].mean().sort_values()

day_of_week
Sunday       6933.231405
Thursday     7405.836735
Friday       7448.230159
Wednesday    7559.373333
Monday       7780.866667
Tuesday      8125.006579
Saturday     8152.975806
Name: TotalSteps, dtype: float64

In [49]:
# Create column weekday_order.  This column gives information of total steps per day

weekday_order = [
    "Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"
]

steps_by_day = (
    df_master_33
    .groupby("day_of_week")["TotalSteps"]
    .mean()
    .reindex(weekday_order)
)

steps_by_day

day_of_week
Monday       7780.866667
Tuesday      8125.006579
Wednesday    7559.373333
Thursday     7405.836735
Friday       7448.230159
Saturday     8152.975806
Sunday       6933.231405
Name: TotalSteps, dtype: float64

In [50]:
# Difference in steps from day to day

step_difference = steps_by_day.diff()

step_difference

day_of_week
Monday               NaN
Tuesday       344.139912
Wednesday    -565.633246
Thursday     -153.536599
Friday         42.393424
Saturday      704.745648
Sunday      -1219.744401
Name: TotalSteps, dtype: float64

In [51]:
# Percent change for steps from day to day

percent_change = steps_by_day.pct_change() * 100

percent_change

day_of_week
Monday             NaN
Tuesday       4.422899
Wednesday    -6.961634
Thursday     -2.031076
Friday        0.572433
Saturday      9.461921
Sunday      -14.960726
Name: TotalSteps, dtype: float64

In [52]:
# Create the weekday_analysis database

weekday_analysis = pd.DataFrame({
    "avg_steps": steps_by_day,
    "step_change": step_difference,
    "percent_change": percent_change
})

weekday_analysis

,avg_steps,step_change,percent_change
day_of_week,,,
Monday,7780.866667,NaN,NaN
Tuesday,8125.006579,344.139912,4.422899
Wednesday,7559.373333,-565.633246,-6.961634
Thursday,7405.836735,-153.536599,-2.031076
Friday,7448.230159,42.393424,0.572433
Saturday,8152.975806,704.745648,9.461921
Sunday,6933.231405,-1219.744401,-14.960726


In [53]:
# Add percent change to weekday_analysis and round percent change to the 2nd decimal

weekday_analysis = weekday_analysis.reset_index()
weekday_analysis["percent_change"] = weekday_analysis["percent_change"].round(2)
weekday_analysis

,day_of_week,avg_steps,step_change,percent_change
0,Monday,7780.866667,NaN,NaN
1,Tuesday,8125.006579,344.139912,4.42
2,Wednesday,7559.373333,-565.633246,-6.96
3,Thursday,7405.836735,-153.536599,-2.03
4,Friday,7448.230159,42.393424,0.57
5,Saturday,8152.975806,704.745648,9.46
6,Sunday,6933.231405,-1219.744401,-14.96


In [54]:
# Save weekday_analysis database

weekday_analysis.to_csv("weekday_analysis.csv", index=False)

In [55]:
# How Active are you?
# Create the ativity category

def classify_activity(steps):
    if steps < 5000:
        return "Sedentary"
    elif steps < 10000:
        return "Moderately Active"
    else:
        return "Active"

df_master_33["ActivityLevel"] = df_master_33["TotalSteps"].apply(classify_activity)    

In [57]:
# Count each category

activity_counts = df_master_33["ActivityLevel"].value_counts()
activity_counts

ActivityLevel
Moderately Active    334
Active               303
Sedentary            303
Name: count, dtype: int64

In [59]:
# Find percent for activity and round to the second decimal

activity_percent = (activity_counts / len(df_master_33)) * 100
activity_percent.round(2)

ActivityLevel
Moderately Active    35.53
Active               32.23
Sedentary            32.23
Name: count, dtype: float64

In [60]:
# Create database for activity

activity_summary = pd.DataFrame({
    "count": activity_counts,
    "percent": activity_percent.round(2)
})

activity_summary

,count,percent
ActivityLevel,,
Moderately Active,334,35.53
Active,303,32.23
Sedentary,303,32.23


In [61]:
# Total steps mean

df_master_33["TotalSteps"].mean()

np.float64(7637.9106382978725)

In [62]:
# test

activity.head(3)

,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,...,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,Date,TotalMinutesAsleep,TotalTimeInBed,TotalSleepRecords,day_of_week,ActivityLevel
0,1503960366,2016-04-12,13162,8.50,8.50,0.0,1.88,0.55,6.06,0.0,...,13,328,728,1985,2016-04-12,327.0,346.0,1.0,Tuesday,Active
1,1503960366,2016-04-13,10735,6.97,6.97,0.0,1.57,0.69,4.71,0.0,...,19,217,776,1797,2016-04-13,384.0,407.0,2.0,Wednesday,Active
2,1503960366,2016-04-14,10460,6.74,6.74,0.0,2.44,0.40,3.91,0.0,...,11,181,1218,1776,2016-04-14,NaN,NaN,NaN,Thursday,Active


In [69]:
# test

print(list(df_master_33.columns))

['Id', 'ActivityDate', 'TotalSteps', 'TotalDistance', 'TrackerDistance', 'LoggedActivitiesDistance', 'VeryActiveDistance', 'ModeratelyActiveDistance', 'LightActiveDistance', 'SedentaryActiveDistance', 'VeryActiveMinutes', 'FairlyActiveMinutes', 'LightlyActiveMinutes', 'SedentaryMinutes', 'Calories', 'Date', 'TotalMinutesAsleep', 'TotalTimeInBed', 'TotalSleepRecords', 'day_of_week', 'ActivityLevel']


In [70]:
# test

print(list(activity_summary.columns))

['count', 'percent']


In [71]:
#test

print(list(weekday_analysis.columns)) 

['day_of_week', 'avg_steps', 'step_change', 'percent_change']


In [74]:
# Combine df_master_33 with weekday_analysis for Tableu visualization

df_combined = df_master_33.merge(
    weekday_analysis,
    on='day_of_week',
    how='left'
)

In [75]:
# Save combined database for visualization.  Go Dolphins!

df_combined.to_csv('df_combined_dashboard.csv', index=False)